In [ ]:
!pip install -q streamlit streamlit-option-menu pyngrok pyjwt bcrypt plotly pandas


In [ ]:
%%writefile app.py
import os, re, sqlite3, jwt, bcrypt, datetime, time, secrets, smtplib, streamlit as st
import plotly.graph_objects as go
from streamlit_option_menu import option_menu
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# 🚀 Force Streamlit Light Theme
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="light"\nprimaryColor="#ffd803"\nbackgroundColor="#f9fcfc"\nsecondaryBackgroundColor="#e3f6f5"\ntextColor="#2d334a"\n')

st.set_page_config(page_title="Infosys Portal", page_icon="⚡", layout="wide", initial_sidebar_state="expanded")

COLORS = {
    "bg_main": "#f9fcfc", "bg_sidebar": "#e3f6f5", "bg_card": "#ffffff", "bg_card_alt": "#bae8e8",
    "text_main": "#2d334a", "text_heading": "#272343", "text_muted": "#64748b",
    "accent": "#ffd803", "accent_hover": "#e6c300", "accent_text": "#272343",
    "border": "#272343", "border_light": "#bae8e8", "success": "#34d399", "danger": "#f87171"
}

JWT_SECRET = os.getenv("JWT_SECRET", "super-secret-infosys-key-2026")
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")
ADMIN_USERNAME = os.getenv("ADMIN_USERNAME", "admin")
ADMIN_PASSWORD = os.getenv("ADMIN_PASSWORD", "Admin@123")
OTP_EXPIRY_MINUTES = 5

# 🚀 Neo-Brutalist CSS + Visible Sidebar Re-Open Button
st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&family=Inter:wght@300;400;500;600&display=swap');
    html, body, .stApp {{ background: {COLORS['bg_main']} !important; font-family: 'Inter', sans-serif !important; color: {COLORS['text_main']} !important; }}

    /* 🚀 DO NOT HIDE HEADER! Only hide footer and bottom decoration so toggle arrow stays visible! */
    footer, div[data-testid="stDecoration"] {{ visibility: hidden !important; display: none !important; }}
    header {{ background: transparent !important; z-index: 999999 !important; }}

    /* 🚀 Highlight Streamlit's Native Sidebar Re-Open Button (Top-Left) as a Yellow Badge! */
    button[kind="header"], div[data-testid="stSidebarCollapsedControl"] button {{
        visibility: visible !important; display: flex !important; opacity: 1 !important;
        background-color: {COLORS['accent']} !important; border: 2px solid {COLORS['border']} !important;
        border-radius: 8px !important; padding: 6px !important; margin: 8px !important;
        box-shadow: 3px 3px 0px {COLORS['border']} !important;
    }}
    button[kind="header"] svg, div[data-testid="stSidebarCollapsedControl"] svg {{
        fill: {COLORS['text_heading']} !important; color: {COLORS['text_heading']} !important; stroke: {COLORS['text_heading']} !important;
    }}

    .block-container {{ padding: 2rem 2.5rem !important; max-width: 1200px; }}
    h1, h2, h3, h4 {{ font-family: 'Poppins', sans-serif !important; color: {COLORS['text_heading']} !important; }}
    label p {{ font-weight: 600 !important; color: {COLORS['text_heading']} !important; }}

    /* Inputs */
    div[data-baseweb="base-input"], div[data-baseweb="select"] > div {{ background-color: transparent !important; border: none !important; }}
    div[data-baseweb="input"], div[data-baseweb="select"] {{ background-color: {COLORS['bg_card']} !important; border: 2px solid {COLORS['border']} !important; border-radius: 10px !important; }}
    div[data-baseweb="input"]:focus-within {{ border-color: {COLORS['accent']} !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; }}
    input, div[data-baseweb="select"] span {{ color: {COLORS['text_main']} !important; -webkit-text-fill-color: {COLORS['text_main']} !important; }}

    /* Buttons */
    div[data-testid="stButton"] button {{
        background-color: {COLORS['accent']} !important; color: {COLORS['accent_text']} !important;
        border: 2px solid {COLORS['border']} !important; border-radius: 10px !important;
        font-family: 'Inter', sans-serif !important; font-weight: 700 !important; font-size: 14px !important;
        height: 48px !important; min-height: 48px !important; white-space: nowrap !important;
        display: flex !important; align-items: center !important; justify-content: center !important;
        padding: 0px 16px !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; width: 100%; transition: all 0.2s ease !important;
    }}
    div[data-testid="stButton"] button:hover {{
        background-color: {COLORS['accent_hover']} !important; transform: translate(-2px, -2px) !important;
        box-shadow: 6px 6px 0px {COLORS['border']} !important;
    }}
    section[data-testid="stSidebar"] {{ background: {COLORS['bg_sidebar']} !important; border-right: 2px solid {COLORS['border']} !important; }}
    .pn-card {{ background: {COLORS['bg_card']}; border: 2px solid {COLORS['border']}; border-radius: 14px; padding: 24px; box-shadow: 4px 4px 0px {COLORS['border_light']}; transition: transform 0.15s ease, box-shadow 0.15s ease; }}
    .pn-card:hover {{ transform: translateY(-3px); box-shadow: 6px 6px 0px {COLORS['border']}; }}

    /* 🚀 Mild UI polish additions */
    div[data-testid="stExpander"] {{ border: 2px solid {COLORS['border']} !important; border-radius: 12px !important; background: {COLORS['bg_card']} !important; box-shadow: 3px 3px 0px {COLORS['border_light']} !important; }}
    div[data-testid="stExpander"] summary {{ font-weight: 700 !important; color: {COLORS['text_heading']} !important; }}
    div[data-testid="stDataFrame"] {{ border: 2px solid {COLORS['border']} !important; border-radius: 10px !important; overflow: hidden; }}
    div[data-testid="stAlert"] {{ border-radius: 10px !important; border: 2px solid {COLORS['border_light']} !important; }}
</style>
""", unsafe_allow_html=True)

def get_db(): return sqlite3.connect("infosys_portal.db", check_same_thread=False)
def hash_txt(t): return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()
def check_txt(t, h): return bcrypt.checkpw(t.encode(), h.encode()) if h else False

with get_db() as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE, email TEXT UNIQUE,
        password_hash TEXT, security_question TEXT, security_answer_hash TEXT)""")
    # 🚀 Admin is NOT a signup account — its credentials are ADMIN_USERNAME / ADMIN_PASSWORD above (Colab Secret or code default)

# 🚀 Field validation helpers
EMAIL_RE = re.compile(r"^[A-Za-z]{2,}[A-Za-z0-9._%+\-]*@[A-Za-z0-9]{2,}(\.[A-Za-z0-9]+)*\.[A-Za-z]{2,}$")
PASSWORD_RE = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[!@#$%^&*()_+\-=\[\]{};':\"\\|,.<>\/?~`]).{8,}$")
def is_valid_email(e): return bool(EMAIL_RE.match(e or ""))
def is_valid_password(p): return bool(PASSWORD_RE.match(p or ""))
PASSWORD_HELP = "Min 8 chars incl. 1 uppercase, 1 lowercase, 1 number & 1 special symbol."

# 🚀 OTP helpers (Gmail SMTP) for Forgot Password → Via OTP
def generate_otp(): return f"{secrets.randbelow(900000) + 100000}"
def make_otp_token(email, otp):
    return jwt.encode({"email": email, "otp_hash": hash_txt(otp), "type": "otp",
                        "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)}, JWT_SECRET, algorithm="HS256")
def verify_otp_token(token, otp_input, email):
    try:
        p = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
        if p.get("email") != email or p.get("type") != "otp": return False, "Security token mismatch."
        if check_txt(otp_input, p["otp_hash"]): return True, "Valid"
        return False, "Invalid 6-digit OTP code."
    except jwt.ExpiredSignatureError: return False, f"⚠️ OTP expired after {OTP_EXPIRY_MINUTES} minutes. Please request a new one."
    except Exception: return False, "Invalid or corrupted verification token."
def send_otp_email(to_email, otp):
    if not EMAIL_ADDRESS or not EMAIL_PASSWORD:
        return False, "Email sender not configured. Add EMAIL_ADDRESS & EMAIL_PASSWORD to Colab Secrets."
    msg = MIMEMultipart("alternative")
    msg["From"] = f"Infosys Portal <{EMAIL_ADDRESS}>"
    msg["To"] = to_email
    msg["Subject"] = "Infosys Portal - Password Reset OTP"
    msg.attach(MIMEText(f"Your OTP is {otp}. It expires in {OTP_EXPIRY_MINUTES} minutes.", "plain"))
    msg.attach(MIMEText(f"""
    <div style="font-family:Arial,sans-serif;text-align:center;padding:20px;">
        <h2 style="color:#272343;">Infosys Portal Verification</h2>
        <p>Your one-time password is:</p>
        <div style="font-size:28px;font-weight:700;letter-spacing:6px;background:#ffd803;
                    display:inline-block;padding:12px 20px;border-radius:10px;border:2px solid #272343;">{otp}</div>
        <p style="color:#64748b;font-size:13px;">Expires in {OTP_EXPIRY_MINUTES} minutes.</p>
    </div>""", "html"))
    try:
        s = smtplib.SMTP("smtp.gmail.com", 587)
        s.starttls()
        s.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        s.sendmail(EMAIL_ADDRESS, to_email, msg.as_string())
        s.quit()
        return True, "OTP sent."
    except Exception as e:
        return False, f"SMTP error: {e}"

def make_jwt(email): return jwt.encode({"email": email, "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)}, JWT_SECRET, algorithm="HS256")
def verify_jwt(token):
    try: return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except: return None

for k, v in [("token", None), ("page", "Login"), ("reset_email", None), ("reset_mode", None), ("otp_token", None), ("otp_verified", False)]:
    if k not in st.session_state: st.session_state[k] = v

def navigate(p): st.session_state.page = p; st.rerun()

def auth_header(title, sub="Intelligent Analytics Portal"):
    st.markdown(f"""
    <div style="text-align:center;padding:1.5rem 0 1rem;">
        <div style="font-size:40px;margin-bottom:10px;">⚡</div>
        <h1 style="font-size:2rem !important;margin:0;">Infosys Portal</h1>
        <p style="color:{COLORS['text_muted']};font-size:14px;margin:4px 0 0;">{sub}</p>
    </div>
    <div style="text-align:center;margin-bottom:1.5rem;"><span style="font-size:1.1rem;font-weight:700;color:{COLORS['text_heading']};">{title}</span></div>
    """, unsafe_allow_html=True)

# ============================================================
# PAGE ROUTING
# ============================================================
if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]:
        st.session_state.page = "Login"

    _, mid, _ = st.columns([1, 1.45, 1])
    with mid:
        if st.session_state.page == "Login":
            auth_header("Sign in to your account")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="••••••••")
            st.markdown("<br>", unsafe_allow_html=True)

            col_l, col_c, col_r = st.columns([1, 1.15, 1.3])
            if col_l.button("Sign In →", use_container_width=True):
                with get_db() as c: r = c.execute("SELECT password_hash FROM users WHERE email=?", (email,)).fetchone()
                if r and check_txt(pwd, r[0]): st.session_state.token = make_jwt(email); navigate("Dashboard")
                else: st.error("❌ Invalid credentials.")
            if col_c.button("Create Account", use_container_width=True): navigate("Signup")
            if col_r.button("Forgot Password", use_container_width=True): navigate("Forgot")

            st.markdown("<br>", unsafe_allow_html=True)
            with st.expander("🛡️ Administrator Login"):
                a_user = st.text_input("Admin Username", placeholder="admin", key="admin_user")
                a_pass = st.text_input("Admin Password", type="password", placeholder="••••••••", key="admin_pass")
                if st.button("Sign In as Admin →", use_container_width=True, key="admin_login_btn"):
                    if a_user == ADMIN_USERNAME and a_pass == ADMIN_PASSWORD:
                        st.session_state.token = make_jwt(ADMIN_USERNAME)
                        navigate("Dashboard")
                    else:
                        st.error("❌ Invalid admin credentials.")

        elif st.session_state.page == "Signup":
            auth_header("Create an account", "Join Infosys Portal today")
            uname = st.text_input("Full name / Username", placeholder="Jane Doe")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="Min. 8 characters")
            confirm_pwd = st.text_input("Confirm password", type="password", placeholder="Re-enter password")
            sq = st.selectbox("Security Question", ["What is your pet name?", "What is your mother's maiden name?", "What is your favourite city?"])
            sa = st.text_input("Your answer", placeholder="Security answer")
            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("Create Account & Login →", use_container_width=True):
                if not uname or not email or not pwd or not confirm_pwd or not sa:
                    st.error("⚠️ Please fill all fields.")
                elif not is_valid_email(email):
                    st.error("❌ Enter a valid email, e.g. ab@cd.ef")
                elif not is_valid_password(pwd):
                    st.error(f"❌ Weak password. {PASSWORD_HELP}")
                elif pwd != confirm_pwd:
                    st.error("❌ Passwords do not match.")
                else:
                    try:
                        with get_db() as c:
                            c.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)", (uname, email, hash_txt(pwd), sq, hash_txt(sa.lower().strip())))
                        st.session_state.token = make_jwt(email)
                        st.success("✅ Account created!")
                        time.sleep(1)
                        navigate("Dashboard")
                    except sqlite3.IntegrityError:
                        st.error("❌ Email or Username already registered.")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Back to Sign In", use_container_width=True): navigate("Login")

        elif st.session_state.page == "Forgot":
            auth_header("Reset your password", "Choose your verification method")
            if not st.session_state.reset_email:
                email = st.text_input("Registered email address", placeholder="you@infosys.com").lower().strip()
                st.markdown("<br>", unsafe_allow_html=True)

                col_sq, col_otp = st.columns(2)
                if col_sq.button("Via Security Question", use_container_width=True):
                    with get_db() as c: r = c.execute("SELECT security_question FROM users WHERE email=?", (email,)).fetchone()
                    if r:
                        st.session_state.reset_email = email
                        st.session_state.sq_p = r[0]
                        st.session_state.reset_mode = "sq"
                        st.rerun()
                    else: st.error("❌ Email not found.")

                if col_otp.button("Via OTP", use_container_width=True):
                    with get_db() as c: r = c.execute("SELECT 1 FROM users WHERE email=?", (email,)).fetchone()
                    if not r:
                        st.error("❌ Email not found.")
                    else:
                        otp = generate_otp()
                        with st.spinner("Sending OTP..."):
                            ok, msg = send_otp_email(email, otp)
                        if ok:
                            st.session_state.reset_email = email
                            st.session_state.reset_mode = "otp"
                            st.session_state.otp_token = make_otp_token(email, otp)
                            st.session_state.otp_verified = False
                            st.success("✅ OTP sent! Check your inbox.")
                            time.sleep(1)
                            st.rerun()
                        else:
                            st.error(f"❌ {msg}")
            else:
                if st.session_state.get("reset_mode") == "sq":
                    st.info(f"❓ **Security Question:** {st.session_state.sq_p}")
                    ans = st.text_input("Your answer").lower().strip()
                    npw = st.text_input("New password (min 8 chars)", type="password")
                    confirm_npw = st.text_input("Confirm new password", type="password")
                    st.markdown("<br>", unsafe_allow_html=True)
                    if st.button("Reset Password →", use_container_width=True):
                        if not is_valid_password(npw):
                            st.error(f"⚠️ {PASSWORD_HELP}")
                        elif npw != confirm_npw:
                            st.error("❌ Passwords do not match.")
                        else:
                            with get_db() as c: r = c.execute("SELECT security_answer_hash FROM users WHERE email=?", (st.session_state.reset_email,)).fetchone()
                            if r and check_txt(ans, r[0]):
                                with get_db() as c: c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                                st.success("✅ Password updated successfully!"); time.sleep(1); st.session_state.reset_email = None; navigate("Login")
                            else: st.error("❌ Incorrect security answer.")

                elif st.session_state.get("reset_mode") == "otp":
                    if not st.session_state.otp_verified:
                        st.info(f"📧 **OTP sent to:** {st.session_state.reset_email} (valid {OTP_EXPIRY_MINUTES} mins)")
                        otp_in = st.text_input("Enter 6-digit OTP", max_chars=6)
                        st.markdown("<br>", unsafe_allow_html=True)
                        if st.button("Verify OTP →", use_container_width=True):
                            ok, msg = verify_otp_token(st.session_state.otp_token, otp_in.strip(), st.session_state.reset_email)
                            if ok:
                                st.session_state.otp_verified = True
                                st.success("✅ OTP verified!"); time.sleep(1); st.rerun()
                            else:
                                st.error(f"❌ {msg}")
                    else:
                        npw = st.text_input("New password (min 8 chars)", type="password", key="otp_npw")
                        confirm_npw = st.text_input("Confirm new password", type="password", key="otp_cnpw")
                        st.markdown("<br>", unsafe_allow_html=True)
                        if st.button("Reset Password →", use_container_width=True, key="otp_reset_btn"):
                            if not is_valid_password(npw):
                                st.error(f"⚠️ {PASSWORD_HELP}")
                            elif npw != confirm_npw:
                                st.error("❌ Passwords do not match.")
                            else:
                                with get_db() as c: c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                                st.success("✅ Password updated successfully!"); time.sleep(1)
                                st.session_state.reset_email = None
                                st.session_state.otp_verified = False
                                navigate("Login")
            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Cancel", use_container_width=True):
                st.session_state.reset_email = None
                st.session_state.reset_mode = None
                st.session_state.otp_verified = False
                navigate("Login")

# ============================================================
# DASHBOARDS (ADMIN vs USER)
# ============================================================
else:
    payload = verify_jwt(st.session_state.token)
    if not payload:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    email = payload["email"]
    if email == ADMIN_USERNAME:
        uname = "Administrator"
    else:
        with get_db() as c: uname = c.execute("SELECT username FROM users WHERE email=?", (email,)).fetchone()[0]

    # 🚀 SIDEBAR MENU
    with st.sidebar:
        st.markdown(f"""
        <div style="padding:16px 8px;text-align:center;">
            <div style="font-size:28px;">⚡</div>
            <div style="font-weight:700;font-size:16px;color:{COLORS['text_heading']};">Infosys Portal</div>
            <div style="font-size:11px;color:{COLORS['text_muted']};">{"Admin Panel" if email==ADMIN_USERNAME else "Intelligent Analytics"}</div>
        </div><hr style="border-color:{COLORS['border_light']};">
        """, unsafe_allow_html=True)

        opts = ["Dashboard", "Settings", "Logout"] if email==ADMIN_USERNAME else ["Dashboard", "Analytics", "Reports", "Logout"]
        menu = option_menu(None, opts, icons=["house", "gear", "box-arrow-right"] if email==ADMIN_USERNAME else ["house", "graph-up", "file-text", "box-arrow-right"],
                           styles={"container": {"background-color": COLORS['bg_sidebar']}, "nav-link-selected": {"background-color": COLORS['accent'], "color": COLORS['accent_text']}})
        if menu == "Logout":
            st.session_state.token = None
            st.session_state.page = "Login"
            st.rerun()

    # 🚀 ADMIN DASHBOARD (hardcoded admin login)
    if email == ADMIN_USERNAME:
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⚡ Infosys Portal</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Admin Control Panel</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">🛡️ {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown(f"""
        <div class="pn-card" style="text-align:center;padding:40px 20px;">
            <h1 style="font-size:32px !important;margin-bottom:10px;">🛡️ Admin Dashboard</h1>
            <p style="color:{COLORS['text_muted']};font-size:16px;font-weight:500;">Welcome to the Administrator area.</p>
        </div>
        """, unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)
        st.markdown(f"<h3 style='color:{COLORS['text_heading']};'>📋 Registered Users</h3>", unsafe_allow_html=True)
        with get_db() as c: all_users = c.execute("SELECT username, email FROM users ORDER BY id DESC").fetchall()
        if all_users:
            import pandas as pd
            st.dataframe(pd.DataFrame(all_users, columns=["Username", "Email"]), use_container_width=True, hide_index=True)
        else:
            st.info("No users have signed up yet.")

    # 🚀 REGULAR USER DASHBOARD
    else:
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⚡ Infosys Portal</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Analytics Dashboard</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">👤 {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        c1, c2, c3, c4 = st.columns(4)
        for col, icon, lbl, val in [(c1, "📄", "Documents Indexed", "128"), (c2, "🔍", "Searches Today", "47"),
                                    (c3, "📊", "Efficiency Score", "98.4%"), (c4, "🛡️", "Security Status", "Secured")]:
            col.markdown(f"""
            <div class="pn-card" style="text-align:center;">
                <div style="font-size:28px;">{icon}</div>
                <div style="font-size:26px;font-weight:700;color:{COLORS['text_heading']};">{val}</div>
                <div style="color:{COLORS['text_muted']};font-size:12px;font-weight:600;">{lbl}</div>
            </div>
            """, unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)
        fig = go.Figure(go.Indicator(mode="gauge+number", value=92, title={"text": "System Health Index", "font": {"color": COLORS['text_heading'], "size": 14}},
                        gauge={"axis": {"range": [0, 100]}, "bar": {"color": COLORS['accent']}, "bgcolor": COLORS['bg_card_alt'], "borderwidth": 1, "bordercolor": COLORS['border']}))
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font={"color": COLORS['text_main'], "family": "Inter"}, height=260, margin=dict(l=10, r=10, t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)


Writing app.py


In [ ]:
import os
import time
import subprocess
from pyngrok import ngrok
from google.colab import userdata

# 1. Retrieve your secret token securely from Colab Secrets
NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
ngrok.set_auth_token(NGROK_TOKEN)

# 1b. Load additional secrets required by the app (JWT signing, Gmail OTP sender, Admin login)
for _key in ["JWT_SECRET", "EMAIL_ADDRESS", "EMAIL_PASSWORD", "ADMIN_USERNAME", "ADMIN_PASSWORD"]:
    try:
        _val = userdata.get(_key)
        if _val: os.environ[_key] = _val
    except Exception:
        pass

# 2. Kill any existing ngrok tunnels or streamlit sessions
ngrok.kill()
!pkill -f streamlit

# 3. Start Streamlit in the background on port 8501
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# 4. Open ngrok tunnel
public_url = ngrok.connect(8501).public_url
print("=" * 60)
print(f"🚀 Infosys Portal Live URL: {public_url}")
print("=" * 60)
print("⏳ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.")

try:
    # Keep the cell active so Ctrl+C can be intercepted
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n" + "🛑" * 30)
    print("Received Ctrl+C / Stop signal. Shutting down...")
    ngrok.kill()
    process.terminate()
    !pkill -f streamlit
    print("✅ Ngrok tunnel closed and Streamlit server stopped gracefully.")
